In [1]:
import pathlib
from decouple import config
from replicate.client import Client

In [2]:

NBS_DIR = pathlib.Path().resolve()
REPO_DIR = NBS_DIR.parent
DATA_DIR = REPO_DIR / "data"
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(exist_ok=True, parents=True)

REPLICATE_API_TOKEN=config("REPLICATE_API_TOKEN")
REPLICATE_MODEL= config("REPLICATE_MODEL", default="aaitorm/superme-aitor-v1")
REPLICATE_MODEL_VERSION= config("REPLICATE_MODEL_VERSION", default="76fd9d004a50354c0d509dda96b7945139f83c9bcd86dc011010ffc11c3d0ea7")

replicate_client= Client(api_token=REPLICATE_API_TOKEN)

In [3]:
model = f"{REPLICATE_MODEL}:{REPLICATE_MODEL_VERSION}"
prompt = "a photo of TOK adult man dressed up for a sports photo shoot"
num_outputs = 2
output_format = "jpg"

input_args = {
    "prompt": prompt,
    "num_outputs": 2,
    "output_format": "jpg",
}

In [4]:
rep_model = replicate_client.models.get(REPLICATE_MODEL)
rep_version = rep_model.versions.get(REPLICATE_MODEL_VERSION)

pred = replicate_client.predictions.create(
    version=rep_version,
    input=input_args
)

In [5]:
pred.id

'7c0da50va9rmt0cw5q88f4mm6c'

In [19]:
pred.status

'starting'

In [20]:
# upstash -> qstash

In [21]:
pred_id = "7c0da50va9rmt0cw5q88f4mm6c"
pred_lookup = replicate_client.predictions.get(pred_id)

In [22]:
pred_lookup.status

'succeeded'

In [23]:
pred_urls = pred_lookup.output
print(pred_urls)

['https://replicate.delivery/xezq/Y9lMb0pDy7a2AZU5ZIBHx88Czz5VsPIn8cq0njBWCeE5CSCLA/out-0.jpg', 'https://replicate.delivery/xezq/9h3dgf6eBrlSlUTeqBOcZ8HiuSH5edR1DltDy4SVx6BLXQSYB/out-1.jpg']


In [24]:
import httpx
import random

session_id = random.randint(1_000, 40_000)
with httpx.Client() as client:
    for i, url in enumerate(pred_urls):
        fname = f"{i}-{session_id}.jpg"
        outpath = GENERATED_DIR / fname
        res = client.get(url)
        res.raise_for_status()
        with open(outpath, 'wb') as f:
            f.write(res.content)